# Deep Learning Project : Generative Models

This notebook implements a **Deep Convolutional GAN (DCGAN)** trained on the real face images from the dataset. The goal is to learn the distribution of real celebrity faces and generate new synthetic face images.

We follow an iterative approach:
- **Option 1** - Baseline DCGAN (standard architecture, default hyperparameters)
- **Option 2** - Improved DCGAN (spectral normalisation, label smoothing, TTUR learning rates, deeper architecture)
- **Option 3** - WGAN-GP (Wasserstein GAN with Gradient Penalty)

## Imports

In [1]:
import os, json, random, shutil
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from glob import glob
from PIL import Image
from scipy import linalg

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import inception_v3
import torchvision.utils as vutils

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Load Dataset

In [2]:
real_images = glob("/kaggle/input/datasets/joanapimenta27/deepfakefacejoana/data/wiki/**/*.jpg", recursive=True)
print(f"Total real images found: {len(real_images)}")

N_TRAIN = 20000
random.shuffle(real_images)
train_paths = real_images[:N_TRAIN]
print(f"Using {len(train_paths)} images for training.")

Total real images found: 30000
Using 20000 images for training.


In [3]:
IMG_SIZE = 128
BATCH_SIZE = 64
LATENT_DIM = 100


class FaceDataset(Dataset):
    """Loads face images and applies transforms. Silently skips corrupt files."""

    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
            return self.transform(img)
        except Exception:
            return self.__getitem__(random.randint(0, len(self) - 1))


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5]),
])

train_dataset = FaceDataset(train_paths, train_transform)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2, pin_memory=True, drop_last=True,
    persistent_workers=True,  # keep workers alive between epochs
    prefetch_factor=2,        # pre-load next batch in background
)
print(f"Dataset size : {len(train_dataset)} images")
print(f"Batches/epoch: {len(train_loader)}")

Dataset size : 20000 images
Batches/epoch: 312


In [4]:
# ── Visualise a sample of training images ────────────────────────────────────
# Load directly — no need to spin up DataLoader workers for a preview
sample_imgs = []
for path in train_paths[:16]:
    img = Image.open(path).convert("RGB")
    sample_imgs.append(train_transform(img))

sample_batch = torch.stack(sample_imgs)
grid = vutils.make_grid(sample_batch, nrow=4, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(8, 8))
plt.title("Training Images (Real)", fontsize=14)
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.axis("off")
plt.tight_layout()
plt.show()

## Utility Functions

In [5]:
def weights_init(m):
    """DCGAN weight initialisation: N(0, 0.02) for Conv/ConvTranspose, 1/0 for BN."""
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def save_checkpoint(state, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(state, path)


def load_checkpoint(path, generator, discriminator, opt_g, opt_d):
    ckpt = torch.load(path, map_location=device)
    generator.load_state_dict(ckpt["generator"])
    discriminator.load_state_dict(ckpt["discriminator"])
    opt_g.load_state_dict(ckpt["opt_g"])
    opt_d.load_state_dict(ckpt["opt_d"])
    return ckpt["epoch"], ckpt["history"]


def denorm(tensor):
    """Convert from [-1,1] back to [0,1] for display."""
    return (tensor * 0.5 + 0.5).clamp(0, 1)

In [6]:
def save_image_grid(generator, fixed_noise, epoch, save_dir, nrow=8):
    """Save generated image grid to file — no display."""
    os.makedirs(save_dir, exist_ok=True)
    generator.eval()
    with torch.no_grad():
        fake = generator(fixed_noise).cpu()
    generator.train()
    grid = vutils.make_grid(fake, nrow=nrow, normalize=True, value_range=(-1, 1))
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.axis("off")
    ax.set_title(f"Generated Faces — Epoch {epoch}", fontsize=13)
    fig.tight_layout()
    path = os.path.join(save_dir, f"epoch_{epoch:04d}.png")
    fig.savefig(path, dpi=100, bbox_inches="tight")
    plt.close(fig)  # never call plt.show()
    print(f"  [saved] {path}", flush=True)


def plot_training_curves(history, title="Training Curves", log_path=None):
    """Save training curves to file — no display."""
    epochs = range(1, len(history["G_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(title, fontsize=14)

    axes[0].plot(epochs, history["G_loss"], label="Generator Loss",     color="steelblue")
    axes[0].plot(epochs, history["D_loss"], label="Discriminator Loss", color="tomato")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("Generator vs Discriminator Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history["D_x"],  label="D(x)   — real",  color="forestgreen")
    axes[1].plot(epochs, history["D_Gz"], label="D(G(z)) — fake", color="darkorange")
    axes[1].axhline(0.5, linestyle="--", color="grey", alpha=0.6, label="Ideal = 0.5")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Mean Discriminator Output")
    axes[1].set_title("Discriminator Confidence")
    axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    if log_path:
        fig.savefig(log_path, dpi=100, bbox_inches="tight")
        print(f"  [saved] {log_path}", flush=True)
    plt.close(fig)


def log_epoch(log_file, epoch, g_loss, d_loss, d_x, d_gz, fid=None):
    fid_str = f", FID {fid:.2f}" if fid is not None else ""
    line = (f"Epoch {epoch:03d} | G_loss {g_loss:.4f}, D_loss {d_loss:.4f} "
            f"| D(x) {d_x:.4f}, D(G(z)) {d_gz:.4f}{fid_str}\n")
    # Write to log file
    with open(log_file, "a") as f:
        f.write(line)
    # Print immediately — flush=True forces it out without buffering
    print(line, end="", flush=True)

In [7]:
# ── FID (Fréchet Inception Distance) ─────────────────────────────────────────
_inception_model = None

def _get_inception():
    global _inception_model
    if _inception_model is None:
        model = inception_v3(pretrained=True, transform_input=False)
        model.fc = nn.Identity()
        model.aux_logits = False
        model.eval()
        _inception_model = model.to(device)
    return _inception_model


@torch.no_grad()
def _extract_features(images_tensor, batch_size=64):
    """images_tensor: N×3×H×W in [-1,1]. Returns numpy array N×2048."""
    model = _get_inception()
    feats = []
    for i in range(0, len(images_tensor), batch_size):
        batch = images_tensor[i:i + batch_size].to(device)
        batch = F.interpolate(batch, size=(299, 299), mode="bilinear",
                              align_corners=False, antialias=True)
        feats.append(model(batch).cpu().numpy())
    return np.concatenate(feats, axis=0)


def compute_fid(real_images_tensor, generator, n_samples=512):
    """Computes FID between real images and n_samples generated images."""
    # Real features
    idx = torch.randperm(len(real_images_tensor))[:n_samples]
    real_feats = _extract_features(real_images_tensor[idx])

    # Fake features
    generator.eval()
    noise = torch.randn(n_samples, LATENT_DIM, device=device)
    fake_imgs = []
    for i in range(0, n_samples, 64):
        fake_imgs.append(generator(noise[i:i + 64]).cpu())
    fake_imgs = torch.cat(fake_imgs, dim=0)
    generator.train()
    fake_feats = _extract_features(fake_imgs)

    # Fréchet distance
    mu1, sigma1 = real_feats.mean(0), np.cov(real_feats, rowvar=False)
    mu2, sigma2 = fake_feats.mean(0), np.cov(fake_feats, rowvar=False)
    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = float(diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean))
    return fid


# ── Pre-collect real image pool — load directly, no DataLoader workers ────────
print("Collecting real image pool for FID evaluation...")
fid_paths = random.sample(train_paths, 512)
fid_real_pool = torch.stack([
    train_transform(Image.open(p).convert("RGB")) for p in fid_paths
])
print(f"FID pool ready: {fid_real_pool.shape}")  # (512, 3, 128, 128)

FID pool ready: torch.Size([512, 3, 128, 128])


---
## Option 1 - Baseline DCGAN

### Motivation
We start with the original DCGAN architecture as described by [Radford et al., 2015](https://arxiv.org/abs/1511.06434). The key design decisions are:
- **Generator**: transposed convolutions with BatchNorm + ReLU, Tanh output
- **Discriminator**: strided convolutions with BatchNorm + LeakyReLU, no sigmoid (BCEWithLogitsLoss handles it)
- **Latent vector** of size 100, projected to a 8×8 spatial map, then upsampled 4× to reach 128×128
- Both networks use standard Adam (lr=0.0002, β₁=0.5) - the β₁ value is specific to GANs to reduce oscillation
- Real labels = 1.0, fake labels = 0.0

This gives us a reproducible baseline against which we can measure the effect of each improvement.

In [8]:
class Generator_v1(nn.Module):
    """Baseline DCGAN generator: latent(100) → 128×128 RGB image.

    Architecture:
      z(100,1,1)  → ConvT(512, k8, s1, p0) → (512, 8, 8)
                  → ConvT(256, k4, s2, p1) → (256,16,16)
                  → ConvT(128, k4, s2, p1) → (128,32,32)
                  → ConvT( 64, k4, s2, p1) → ( 64,64,64)
                  → ConvT(  3, k4, s2, p1) → (  3,128,128) + Tanh
    """
    def __init__(self, latent_dim=LATENT_DIM, ngf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 8, 8, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8), nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),

            nn.ConvTranspose2d(ngf, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z.view(z.size(0), z.size(1), 1, 1))


class Discriminator_v1(nn.Module):
    """Baseline DCGAN discriminator: 128×128 RGB → real/fake logit.

    Architecture:
      (3,128,128)  → Conv( 64, k4, s2, p1) + LeakyReLU → ( 64, 64, 64)
                   → Conv(128, k4, s2, p1) + BN + LeakyReLU → (128, 32, 32)
                   → Conv(256, k4, s2, p1) + BN + LeakyReLU → (256, 16, 16)
                   → Conv(512, k4, s2, p1) + BN + LeakyReLU → (512,  8,  8)
                   → Conv(  1, k8, s1, p0)                  → scalar logit
    """
    def __init__(self, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8), nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 8, 1, 8, 1, 0, bias=False),
        )

    def forward(self, x):
        return self.net(x).view(-1)


# Quick sanity check
with torch.no_grad():
    _z  = torch.randn(4, LATENT_DIM)
    _gv1 = Generator_v1()
    _dv1 = Discriminator_v1()
    _fake = _gv1(_z)
    print("Generator_v1   output shape:", _fake.shape)   # (4, 3, 128, 128)
    print("Discriminator_v1 output shape:", _dv1(_fake).shape)  # (4,)

Generator_v1   output shape: torch.Size([4, 3, 128, 128])
Discriminator_v1 output shape: torch.Size([4])


In [9]:
def train_dcgan(
    generator, discriminator,
    opt_g, opt_d,
    loader,
    epochs,
    experiment_name,
    base_log_dir="experiment_logs",
    fixed_noise=None,
    label_smoothing=0.0,    # 0.0 = no smoothing (Iteration 1)
    n_critic=1,             # how many D steps per G step
    fid_every=5,            # compute FID every N epochs (expensive)
    resume=True,
):
    """
    Generic DCGAN training loop shared by both iterations.
    Returns history dict with per-epoch metrics.
    """
    log_dir  = os.path.join(base_log_dir, experiment_name)
    img_dir  = os.path.join(log_dir, "images")
    ckpt_path = os.path.join(log_dir, "checkpoint.pt")
    log_file  = os.path.join(log_dir, "logs.txt")
    os.makedirs(img_dir, exist_ok=True)

    if fixed_noise is None:
        fixed_noise = torch.randn(64, LATENT_DIM, device=device)

    criterion = nn.BCEWithLogitsLoss()

    history = {"G_loss": [], "D_loss": [], "D_x": [], "D_Gz": [], "FID": []}
    start_epoch = 1

    # ── Resume if checkpoint exists ───────────────────────────────────────────
    if resume and os.path.exists(ckpt_path):
        print(f"[RESUME] Loading checkpoint from {ckpt_path}")
        start_epoch, history = load_checkpoint(
            ckpt_path, generator, discriminator, opt_g, opt_d)
        start_epoch += 1
        print(f"  Resuming from epoch {start_epoch}")

    generator.train()
    discriminator.train()

    for epoch in range(start_epoch, epochs + 1):
        g_losses, d_losses, dx_vals, dgz_vals = [], [], [], []

        for i, real_imgs in enumerate(loader):
            real_imgs = real_imgs.to(device)
            bs = real_imgs.size(0)

            # ── Labels (with optional smoothing) ──────────────────────────────
            real_label = torch.full((bs,), 1.0 - label_smoothing, device=device)
            fake_label = torch.zeros(bs, device=device)

            # ── Train Discriminator ───────────────────────────────────────────
            for _ in range(n_critic):
                discriminator.zero_grad()
                # Real
                out_real = discriminator(real_imgs)
                d_loss_real = criterion(out_real, real_label)
                # Fake
                noise = torch.randn(bs, LATENT_DIM, device=device)
                fake_imgs = generator(noise).detach()
                out_fake = discriminator(fake_imgs)
                d_loss_fake = criterion(out_fake, fake_label)
                d_loss = d_loss_real + d_loss_fake
                d_loss.backward()
                opt_d.step()

            # D(x) and D(G(z)) for logging (before G update)
            dx_vals.append(torch.sigmoid(out_real).mean().item())
            dgz_vals.append(torch.sigmoid(out_fake).mean().item())

            # ── Train Generator ───────────────────────────────────────────────
            generator.zero_grad()
            noise = torch.randn(bs, LATENT_DIM, device=device)
            fake_imgs = generator(noise)
            out_fake2 = discriminator(fake_imgs)
            # Generator wants discriminator to believe fakes are real
            g_loss = criterion(out_fake2, torch.ones(bs, device=device))
            g_loss.backward()
            opt_g.step()

            g_losses.append(g_loss.item())
            d_losses.append(d_loss.item())

        # ── Epoch-level metrics ───────────────────────────────────────────────
        mean_g  = np.mean(g_losses)
        mean_d  = np.mean(d_losses)
        mean_dx = np.mean(dx_vals)
        mean_dgz = np.mean(dgz_vals)

        # FID (computed every fid_every epochs — it is slow)
        fid = None
        if epoch % fid_every == 0:
            fid = compute_fid(fid_real_pool, generator)
            history["FID"].append((epoch, fid))

        history["G_loss"].append(mean_g)
        history["D_loss"].append(mean_d)
        history["D_x"].append(mean_dx)
        history["D_Gz"].append(mean_dgz)

        log_epoch(log_file, epoch, mean_g, mean_d, mean_dx, mean_dgz, fid)

        # Save image grid every 5 epochs
        if epoch % 5 == 0 or epoch == 1:
            save_image_grid(generator, fixed_noise, epoch, img_dir)

        # Checkpoint every 10 epochs
        if epoch % 10 == 0:
            save_checkpoint({
                "epoch": epoch,
                "generator":     generator.state_dict(),
                "discriminator": discriminator.state_dict(),
                "opt_g":         opt_g.state_dict(),
                "opt_d":         opt_d.state_dict(),
                "history":       history,
            }, ckpt_path)

    # Save final history to JSON
    with open(os.path.join(log_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    return generator, discriminator, history

In [10]:
# ── Run Iteration 1 ───────────────────────────────────────────────────────────
EPOCHS_V1 = 250
LR_V1     = 2e-4

gen_v1  = Generator_v1().to(device)
disc_v1 = Discriminator_v1().to(device)
gen_v1.apply(weights_init)
disc_v1.apply(weights_init)

opt_g_v1 = torch.optim.Adam(gen_v1.parameters(),  lr=LR_V1, betas=(0.5, 0.999))
opt_d_v1 = torch.optim.Adam(disc_v1.parameters(), lr=LR_V1, betas=(0.5, 0.999))

fixed_noise = torch.randn(64, LATENT_DIM, device=device)

print(f"Generator     parameters: {sum(p.numel() for p in gen_v1.parameters()):,}")
print(f"Discriminator parameters: {sum(p.numel() for p in disc_v1.parameters()):,}")

gen_v1, disc_v1, history_v1 = train_dcgan(
    gen_v1, disc_v1,
    opt_g_v1, opt_d_v1,
    train_loader,
    epochs=EPOCHS_V1,
    experiment_name="DCGAN_v1",
    fixed_noise=fixed_noise,
    label_smoothing=0.0,
    fid_every=10,
)

Generator     parameters: 6,034,304
Discriminator parameters: 2,790,144


Epoch 001 | G_loss 10.3171, D_loss 0.9859 | D(x) 0.7972, D(G(z)) 0.2094


  [saved] experiment_logs/DCGAN_v1/images/epoch_0001.png


Epoch 002 | G_loss 5.1966, D_loss 0.6404 | D(x) 0.8276, D(G(z)) 0.1697


Epoch 003 | G_loss 5.3816, D_loss 0.5895 | D(x) 0.8359, D(G(z)) 0.1588


...

Epoch 247 | G_loss 6.1103, D_loss 0.0921 | D(x) 0.9690, D(G(z)) 0.0309


Epoch 248 | G_loss 5.9919, D_loss 0.2272 | D(x) 0.9458, D(G(z)) 0.0550


Epoch 249 | G_loss 6.3729, D_loss 0.0438 | D(x) 0.9809, D(G(z)) 0.0187


Epoch 250 | G_loss 5.5718, D_loss 0.4077 | D(x) 0.9189, D(G(z)) 0.0814, FID 140.86


  [saved] experiment_logs/DCGAN_v1/images/epoch_0250.png


In [11]:
torch.save(gen_v1.state_dict(), "experiment_logs/DCGAN_v1/generator_final.pt")
print("Generator v1 saved!")

Generator v1 saved!


### Iteration 1 - Results

In [12]:
# Load from JSON if not in memory
v1_history_path = os.path.join("experiment_logs", "DCGAN_v1", "history.json")
if 'history_v1' not in locals() and os.path.exists(v1_history_path):
    with open(v1_history_path) as f:
        history_v1 = json.load(f)

plot_training_curves(
    history_v1,
    title="Iteration 1 — Baseline DCGAN Training Curves",
    log_path=os.path.join("experiment_logs", "DCGAN_v1", "training_curves.png")
)

# FID progression
if history_v1["FID"]:
    fid_epochs, fid_scores = zip(*history_v1["FID"])
    plt.figure(figsize=(7, 3))
    plt.plot(fid_epochs, fid_scores, marker="o", color="purple")
    plt.title("Iteration 1 — FID Score over Training")
    plt.xlabel("Epoch"); plt.ylabel("FID (lower is better)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join("experiment_logs", "DCGAN_v1", "fid.png"), dpi=100)
    plt.show()
    print(f"\nFinal FID (Iteration 1): {fid_scores[-1]:.2f}")

  [saved] experiment_logs/DCGAN_v1/training_curves.png



Final FID (Iteration 1): 140.86


In [13]:
save_image_grid(gen_v1, fixed_noise, epoch=9999,
                save_dir=os.path.join("experiment_logs", "DCGAN_v1", "images"))

  [saved] experiment_logs/DCGAN_v1/images/epoch_9999.png


---
## Option 2 — Improved DCGAN

### Architecture Changes

In [14]:
def spectral_norm(module):
    """Convenience wrapper for nn.utils.spectral_norm."""
    return nn.utils.spectral_norm(module)


class Generator_v2(nn.Module):
    """Improved DCGAN generator: larger capacity (ngf=128), same topology.

    The increase in ngf gives the generator more filters per layer, allowing
    it to model finer facial textures at 128×128 resolution.
    """
    def __init__(self, latent_dim=LATENT_DIM, ngf=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 8, 8, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8), nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),

            nn.ConvTranspose2d(ngf, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z.view(z.size(0), z.size(1), 1, 1))


class Discriminator_v2(nn.Module):
    """Improved DCGAN discriminator with Spectral Normalisation.

    Spectral Norm (SN) replaces BatchNorm in the discriminator. SN controls
    the spectral radius (largest singular value) of each weight matrix,
    enforcing a 1-Lipschitz constraint on D. This prevents D from becoming
    arbitrarily confident, which in turn keeps the generator's gradient
    signal healthy throughout training.
    """
    def __init__(self, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            spectral_norm(nn.Conv2d(3, ndf, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),

            spectral_norm(nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),

            spectral_norm(nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),

            spectral_norm(nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),

            spectral_norm(nn.Conv2d(ndf * 8, 1, 8, 1, 0, bias=False)),
        )

    def forward(self, x):
        return self.net(x).view(-1)


# Sanity check
with torch.no_grad():
    _gv2 = Generator_v2()
    _dv2 = Discriminator_v2()
    _fake2 = _gv2(torch.randn(4, LATENT_DIM))
    print("Generator_v2   output shape:", _fake2.shape)
    print("Discriminator_v2 output shape:", _dv2(_fake2).shape)
    print(f"Generator_v2   params: {sum(p.numel() for p in _gv2.parameters()):,}")
    print(f"Discriminator_v2 params: {sum(p.numel() for p in _dv2.parameters()):,}")

Generator_v2   output shape: torch.Size([4, 3, 128, 128])
Discriminator_v2 output shape: torch.Size([4])
Generator_v2   params: 17,573,632
Discriminator_v2 params: 2,788,352


In [15]:
# ── Run Iteration 2 ───────────────────────────────────────────────────────────
shutil.rmtree("experiment_logs/DCGAN_v2", ignore_errors=True)

EPOCHS_V2          = 100
LR_D_V2            = 1e-4   # was 2e-4 — slow D down more
LR_G_V2            = 2e-4   # was 1e-4 — speed G up (flip the TTUR)
LABEL_SMOOTHING_V2 = 0.15   # was 0.05 — more aggressive smoothing

gen_v2  = Generator_v2().to(device)
disc_v2 = Discriminator_v2().to(device)
gen_v2.apply(weights_init)
disc_v2.apply(weights_init)

opt_g_v2 = torch.optim.Adam(gen_v2.parameters(),  lr=LR_G_V2, betas=(0.5, 0.999))
opt_d_v2 = torch.optim.Adam(disc_v2.parameters(), lr=LR_D_V2, betas=(0.5, 0.999))

print(f"Generator_v2     parameters: {sum(p.numel() for p in gen_v2.parameters()):,}")
print(f"Discriminator_v2 parameters: {sum(p.numel() for p in disc_v2.parameters()):,}")

gen_v2, disc_v2, history_v2 = train_dcgan(
    gen_v2, disc_v2,
    opt_g_v2, opt_d_v2,
    train_loader,
    epochs=EPOCHS_V2,
    experiment_name="DCGAN_v2",
    fixed_noise=fixed_noise,
    label_smoothing=LABEL_SMOOTHING_V2,
    fid_every=10,
)

Generator_v2     parameters: 17,573,632
Discriminator_v2 parameters: 2,788,352


Epoch 001 | G_loss 0.9095, D_loss 1.1478 | D(x) 0.7537, D(G(z)) 0.4555


  [saved] experiment_logs/DCGAN_v2/images/epoch_0001.png


Epoch 002 | G_loss 0.7702, D_loss 1.3066 | D(x) 0.6214, D(G(z)) 0.5107


Epoch 003 | G_loss 0.7585, D_loss 1.2851 | D(x) 0.6391, D(G(z)) 0.5064


...

Epoch 059 | G_loss 0.6870, D_loss 1.1913 | D(x) 0.7627, D(G(z)) 0.5196


Epoch 060 | G_loss 0.6880, D_loss 1.1902 | D(x) 0.7607, D(G(z)) 0.5187, FID 271.81


  [saved] experiment_logs/DCGAN_v2/images/epoch_0060.png


Epoch 061 | G_loss 0.6855, D_loss 1.1956 | D(x) 0.7588, D(G(z)) 0.5209


Epoch 062 | G_loss 0.6857, D_loss 1.2049 | D(x) 0.7502, D(G(z)) 0.5228


In [ ]:
torch.save(gen_v2.state_dict(), "experiment_logs/DCGAN_v2/generator_final.pt")
print("Generator v2 saved!")

### Option 2 — Results

In [ ]:
v2_history_path = os.path.join("experiment_logs", "DCGAN_v2", "history.json")
if 'history_v2' not in locals() and os.path.exists(v2_history_path):
    with open(v2_history_path) as f:
        history_v2 = json.load(f)

plot_training_curves(
    history_v2,
    title="Iteration 2 — Improved DCGAN Training Curves",
    log_path=os.path.join("experiment_logs", "DCGAN_v2", "training_curves.png")
)

if history_v2["FID"]:
    fid_epochs, fid_scores = zip(*history_v2["FID"])
    plt.figure(figsize=(7, 3))
    plt.plot(fid_epochs, fid_scores, marker="o", color="purple")
    plt.title("Iteration 2 — FID Score over Training")
    plt.xlabel("Epoch"); plt.ylabel("FID (lower is better)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join("experiment_logs", "DCGAN_v2", "fid.png"), dpi=100)
    plt.show()
    print(f"\nFinal FID (Iteration 2): {fid_scores[-1]:.2f}")

In [ ]:
save_image_grid(gen_v2, fixed_noise, epoch=9999,
                save_dir=os.path.join("experiment_logs", "DCGAN_v2", "images"))

---
## Option 3 — WGAN-GP (Wasserstein GAN with Gradient Penalty)

### Motivation

Onptions 1 and 2 both suffered from a fundamental problem: the **discriminator becoming too confident**, which either:
- Starves the generator of useful gradients (Iteration 1: D(G(z)) ≈ 0.20)
- Triggers mode collapse and grey images (Iteration 2: partial collapse after epoch 10)

Both problems have the **same root cause** - the standard binary cross-entropy loss used in DCGAN saturates when the discriminator is too good. Once the discriminator confidently rejects all fakes, the generator receives a gradient of ~0 and stops learning.

**Wasserstein GAN with Gradient Penalty (WGAN-GP)** solves this by:

| Change | Effect |
|---|---|
| **Wasserstein loss** instead of BCE | Loss never saturates - always provides a meaningful gradient signal to G |
| **Gradient Penalty** instead of weight clipping | Enforces the 1-Lipschitz constraint smoothly, without the instability of clipping |
| **No sigmoid** in discriminator output | D outputs a real-valued score (not a probability), so it can't "saturate" |
| **n_critic = 5** | Train D 5× per G step - D must be well-trained for the Wasserstein distance to be meaningful |

The key insight: in WGAN-GP, the discriminator (called the **critic**) is *supposed* to keep improving - a better critic gives a better gradient signal to G. This is the opposite of standard GANs where an overpowered D is the problem.

### Architecture

We reuse the same Generator_v2 architecture (ngf=128) from Iteration 2. The only architectural change is in the **Critic** (discriminator):
- No BatchNorm (incompatible with gradient penalty - it creates correlations between samples in a batch)
- No sigmoid output - outputs a raw score
- Uses Instance Normalisation instead, which normalises per-sample rather than per-batch

In [ ]:
class Critic_v3(nn.Module):
    """WGAN-GP Critic: 128x128 RGB -> scalar score (not probability).

    Key differences from Discriminator_v1/v2:
    - No BatchNorm (incompatible with gradient penalty)
    - No sigmoid output — raw score, can be any real number
    - Uses Instance Norm instead (normalises per-sample, compatible with GP)
    - Spectral Norm retained for additional stability
    """
    def __init__(self, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            # Block 1 — no norm on first layer (standard practice)
            nn.utils.spectral_norm(nn.Conv2d(3, ndf, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 2
            nn.utils.spectral_norm(nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False)),
            nn.InstanceNorm2d(ndf * 2, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 3
            nn.utils.spectral_norm(nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False)),
            nn.InstanceNorm2d(ndf * 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 4
            nn.utils.spectral_norm(nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False)),
            nn.InstanceNorm2d(ndf * 8, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            # Output — raw score, no sigmoid
            nn.utils.spectral_norm(nn.Conv2d(ndf * 8, 1, 8, 1, 0, bias=False)),
        )

    def forward(self, x):
        return self.net(x).view(-1)


# Sanity check
with torch.no_grad():
    _gv3 = Generator_v2()   # reuse same generator architecture
    _cv3 = Critic_v3()
    _fake3 = _gv3(torch.randn(4, LATENT_DIM))
    print("Generator_v3 (reused v2) output shape:", _fake3.shape)
    print("Critic_v3 output shape:", _cv3(_fake3).shape)
    print(f"Generator params : {sum(p.numel() for p in _gv3.parameters()):,}")
    print(f"Critic params    : {sum(p.numel() for p in _cv3.parameters()):,}")

### WGAN-GP Loss Functions

The Wasserstein loss is simpler than BCE:
- **Critic loss**: `mean(critic(fake)) - mean(critic(real))` + gradient penalty
- **Generator loss**: `-mean(critic(fake))` - maximise the critic score on fakes

The gradient penalty enforces the 1-Lipschitz constraint by penalising gradients that deviate from norm 1.

In [ ]:
def gradient_penalty(critic, real_imgs, fake_imgs, device, lambda_gp=10):
    """
    Computes the gradient penalty for WGAN-GP.

    Creates interpolated images between real and fake, passes them through
    the critic, and penalises gradients that deviate from norm 1.
    lambda_gp=10 is the standard value from the original paper.
    """
    bs = real_imgs.size(0)
    # Random interpolation coefficient per image in batch
    alpha = torch.rand(bs, 1, 1, 1, device=device)
    # Interpolate between real and fake
    interpolated = (alpha * real_imgs + (1 - alpha) * fake_imgs.detach()).requires_grad_(True)
    # Critic score on interpolated images
    d_interp = critic(interpolated)
    # Compute gradients w.r.t. interpolated images
    gradients = torch.autograd.grad(
        outputs=d_interp,
        inputs=interpolated,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True,
        retain_graph=True,
    )[0]
    # Flatten gradients and compute their L2 norm
    gradients = gradients.view(bs, -1)
    gradient_norm = gradients.norm(2, dim=1)
    # Penalty: (||gradient|| - 1)^2
    gp = lambda_gp * ((gradient_norm - 1) ** 2).mean()
    return gp

In [ ]:
def train_wgangp(
    generator, critic,
    opt_g, opt_c,
    loader,
    epochs,
    experiment_name,
    base_log_dir="experiment_logs",
    fixed_noise=None,
    n_critic=5,        # train critic 5x per generator step
    lambda_gp=10,      # gradient penalty weight
    fid_every=10,
    resume=True,
):
    """
    WGAN-GP training loop.

    Key differences from train_dcgan:
    - No BCE loss — uses Wasserstein loss
    - Critic trained n_critic times per generator step
    - Gradient penalty added to critic loss
    - Tracks W-distance (should increase then stabilise)
    """
    log_dir   = os.path.join(base_log_dir, experiment_name)
    img_dir   = os.path.join(log_dir, "images")
    ckpt_path = os.path.join(log_dir, "checkpoint.pt")
    log_file  = os.path.join(log_dir, "logs.txt")
    os.makedirs(img_dir, exist_ok=True)

    if fixed_noise is None:
        fixed_noise = torch.randn(64, LATENT_DIM, device=device)

    history = {"G_loss": [], "C_loss": [], "W_dist": [], "GP": [], "FID": []}
    start_epoch = 1

    # Resume from checkpoint if available
    if resume and os.path.exists(ckpt_path):
        print(f"[RESUME] Loading checkpoint from {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=device)
        generator.load_state_dict(ckpt["generator"])
        critic.load_state_dict(ckpt["critic"])
        opt_g.load_state_dict(ckpt["opt_g"])
        opt_c.load_state_dict(ckpt["opt_c"])
        start_epoch = ckpt["epoch"] + 1
        history = ckpt["history"]
        print(f"  Resuming from epoch {start_epoch}")

    generator.train()
    critic.train()

    for epoch in range(start_epoch, epochs + 1):
        g_losses, c_losses, w_dists, gps = [], [], [], []

        for i, real_imgs in enumerate(loader):
            real_imgs = real_imgs.to(device)
            bs = real_imgs.size(0)

            # ── Train Critic n_critic times ───────────────────────────────────
            for _ in range(n_critic):
                critic.zero_grad()
                noise = torch.randn(bs, LATENT_DIM, device=device)
                fake_imgs = generator(noise).detach()

                # Wasserstein loss: maximise critic(real) - critic(fake)
                c_real = critic(real_imgs).mean()
                c_fake = critic(fake_imgs).mean()
                gp     = gradient_penalty(critic, real_imgs, fake_imgs, device, lambda_gp)
                c_loss = c_fake - c_real + gp   # minimise this
                c_loss.backward()
                opt_c.step()

            # Log critic metrics (W-distance is real - fake, higher = better)
            w_dist = (c_real - c_fake).item()
            w_dists.append(w_dist)
            c_losses.append(c_loss.item())
            gps.append(gp.item())

            # ── Train Generator ───────────────────────────────────────────────
            generator.zero_grad()
            noise = torch.randn(bs, LATENT_DIM, device=device)
            fake_imgs = generator(noise)
            # Generator wants critic to score fakes as high as possible
            g_loss = -critic(fake_imgs).mean()
            g_loss.backward()
            opt_g.step()
            g_losses.append(g_loss.item())

        # ── Epoch metrics ─────────────────────────────────────────────────────
        mean_g  = np.mean(g_losses)
        mean_c  = np.mean(c_losses)
        mean_w  = np.mean(w_dists)
        mean_gp = np.mean(gps)

        fid = None
        if epoch % fid_every == 0:
            fid = compute_fid(fid_real_pool, generator)
            history["FID"].append((epoch, fid))

        history["G_loss"].append(mean_g)
        history["C_loss"].append(mean_c)
        history["W_dist"].append(mean_w)
        history["GP"].append(mean_gp)

        # Log
        fid_str = f", FID {fid:.2f}" if fid is not None else ""
        line = (f"Epoch {epoch:03d} | G_loss {mean_g:.4f}, C_loss {mean_c:.4f} "
                f"| W_dist {mean_w:.4f}, GP {mean_gp:.4f}{fid_str}\n")
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        with open(log_file, "a") as f:
            f.write(line)
        print(line, end="", flush=True)

        if epoch % 5 == 0 or epoch == 1:
            save_image_grid(generator, fixed_noise, epoch, img_dir)

        if epoch % 10 == 0:
            torch.save({
                "epoch": epoch,
                "generator": generator.state_dict(),
                "critic":    critic.state_dict(),
                "opt_g":     opt_g.state_dict(),
                "opt_c":     opt_c.state_dict(),
                "history":   history,
            }, ckpt_path)

    with open(os.path.join(log_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    return generator, critic, history

In [ ]:
# ── Run Iteration 3 ───────────────────────────────────────────────────────────
shutil.rmtree("experiment_logs/DCGAN_v3", ignore_errors=True)

EPOCHS_V3   = 30
LR_G_V3     = 1e-4    # WGAN-GP paper recommends 1e-4 for both
LR_C_V3     = 1e-4
N_CRITIC    = 5       # train critic 5x per G step (standard WGAN-GP)
LAMBDA_GP   = 10      # gradient penalty weight (standard value)

gen_v3    = Generator_v2().to(device)   # reuse same generator architecture
critic_v3 = Critic_v3().to(device)
gen_v3.apply(weights_init)
critic_v3.apply(weights_init)

# WGAN-GP uses Adam with low beta1 (0.0 or 0.5)
opt_g_v3 = torch.optim.Adam(gen_v3.parameters(),    lr=LR_G_V3, betas=(0.0, 0.9))
opt_c_v3 = torch.optim.Adam(critic_v3.parameters(), lr=LR_C_V3, betas=(0.0, 0.9))

print(f"Generator_v3  parameters: {sum(p.numel() for p in gen_v3.parameters()):,}")
print(f"Critic_v3     parameters: {sum(p.numel() for p in critic_v3.parameters()):,}")

gen_v3, critic_v3, history_v3 = train_wgangp(
    gen_v3, critic_v3,
    opt_g_v3, opt_c_v3,
    train_loader,
    epochs=EPOCHS_V3,
    experiment_name="DCGAN_v3",
    fixed_noise=fixed_noise,
    n_critic=N_CRITIC,
    lambda_gp=LAMBDA_GP,
    fid_every=10,
)

In [ ]:
# Save final generator
torch.save(gen_v3.state_dict(), "experiment_logs/DCGAN_v3/generator_final.pt")
print("Generator v3 saved!")

### Iteration 3 — Results

In [ ]:
# Load from JSON if not in memory
v3_history_path = os.path.join("experiment_logs", "DCGAN_v3", "history.json")
if 'history_v3' not in locals() and os.path.exists(v3_history_path):
    with open(v3_history_path) as f:
        history_v3 = json.load(f)

# Training curves — WGAN-GP has different metrics than DCGAN
epochs_v3 = range(1, len(history_v3["G_loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle("Iteration 3 — WGAN-GP Training Curves", fontsize=14)

# G and C loss
axes[0].plot(epochs_v3, history_v3["G_loss"], label="Generator Loss",  color="steelblue")
axes[0].plot(epochs_v3, history_v3["C_loss"], label="Critic Loss",     color="tomato")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Generator vs Critic Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Wasserstein distance — should increase then stabilise
# A higher W-distance means the critic is more certain real > fake
axes[1].plot(epochs_v3, history_v3["W_dist"], color="forestgreen")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("W-distance")
axes[1].set_title("Wasserstein Distance (higher = better separation)")
axes[1].grid(True, alpha=0.3)

# Gradient penalty — should stay roughly constant if training is stable
axes[2].plot(epochs_v3, history_v3["GP"], color="purple")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Gradient Penalty")
axes[2].set_title("Gradient Penalty (should be stable)")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join("experiment_logs", "DCGAN_v3", "training_curves.png"), dpi=100)
plt.close()
print("Training curves saved.")

# FID
if history_v3["FID"]:
    fid_epochs, fid_scores = zip(*history_v3["FID"])
    plt.figure(figsize=(7, 3))
    plt.plot(fid_epochs, fid_scores, marker="o", color="purple")
    plt.title("Iteration 3 — FID Score over Training")
    plt.xlabel("Epoch"); plt.ylabel("FID (lower is better)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join("experiment_logs", "DCGAN_v3", "fid.png"), dpi=100)
    plt.close()
    print(f"\nFinal FID (Iteration 3): {fid_scores[-1]:.2f}")

In [ ]:
save_image_grid(gen_v3, fixed_noise, epoch=9999,
                save_dir=os.path.join("experiment_logs", "DCGAN_v3", "images"))

### Option 3 - Analysis & Comparison


In [ ]:
# ── Full 3-way FID comparison ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Final Comparison: All Three Iterations", fontsize=14)

# FID comparison
if history_v1["FID"] and history_v2["FID"] and history_v3["FID"]:
    fe1, fs1 = zip(*history_v1["FID"])
    fe2, fs2 = zip(*history_v2["FID"])
    fe3, fs3 = zip(*history_v3["FID"])
    axes[0].plot(fe1, fs1, marker="o", label="Iter 1 — Baseline DCGAN",    color="steelblue")
    axes[0].plot(fe2, fs2, marker="s", label="Iter 2 — Improved DCGAN",   color="darkorange")
    axes[0].plot(fe3, fs3, marker="^", label="Iter 3 — WGAN-GP",           color="forestgreen")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("FID (lower is better)")
    axes[0].set_title("FID Score — All Iterations")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Summary table as bar chart
iterations  = ["Iter 1\nBaseline DCGAN", "Iter 2\nImproved DCGAN", "Iter 3\nWGAN-GP"]
final_fids  = [fs1[-1], fs2[-1], fs3[-1]]
colors      = ["steelblue", "darkorange", "forestgreen"]
bars = axes[1].bar(iterations, final_fids, color=colors, width=0.5)
for bar, fid in zip(bars, final_fids):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f"{fid:.1f}", ha="center", va="bottom", fontweight="bold")
axes[1].set_ylabel("Final FID (lower is better)")
axes[1].set_title("Final FID at Epoch 30")
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(os.path.join("experiment_logs", "final_comparison.png"), dpi=100)
plt.close()
print("Final comparison saved.")

# Print summary
print("\n" + "=" * 55)
print("FINAL RESULTS SUMMARY")
print("=" * 55)
print(f"  Iter 1 — Baseline DCGAN  : FID {fs1[-1]:.2f}")
print(f"  Iter 2 — Improved DCGAN  : FID {fs2[-1]:.2f}")
print(f"  Iter 3 — WGAN-GP         : FID {fs3[-1]:.2f}")
print("=" * 55)

In [ ]:
# ── Visual comparison of all 3 iterations ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Generated Faces — Final Epoch (All Iterations)", fontsize=14)

for ax, gen, title in [
    (axes[0], gen_v1, "Iter 1\nBaseline DCGAN"),
    (axes[1], gen_v2, "Iter 2\nImproved DCGAN"),
    (axes[2], gen_v3, "Iter 3\nWGAN-GP"),
]:
    gen.eval()
    with torch.no_grad():
        fake = gen(fixed_noise[:16]).cpu()
    grid = vutils.make_grid(fake, nrow=4, normalize=True, value_range=(-1, 1))
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.set_title(title, fontsize=12)
    ax.axis("off")

plt.tight_layout()
plt.savefig(os.path.join("experiment_logs", "visual_comparison_all.png"), dpi=100)
plt.close()
print("Visual comparison saved.")